# ZelusBench: Sustained Attention — Medium**Medium chains (8-10 hops). Moderate tracking difficulty.**Varies chain depth while randomizing background conditions(distractors, transforms, dimensionality, point types).LINEAR structure pinned to guarantee exact depth targeting.Backgrounds are deterministic per scenario index — shared across all levels.- Depths: [8, 9, 10]- 10 scenarios per depth = 30 scenarios

In [ ]:
import kaggle_benchmarks as kbench
import numpy as np
import random
import re
import time

from zelusbench.scenarios.config import (
    ScenarioConfig, DAGStructure, QueryType,
    DistractorLevel, TransformDensity,
)
from zelusbench.scenarios.generator import ScenarioGenerator
from zelusbench.evaluation.scorer import score_query, score_response




DEPTHS = [8, 9, 10]
SEEDS = 10
TOTAL = len(DEPTHS) * SEEDS

print(f"ZelusBench Sustained Attention — Medium")
print(f"Depths: {DEPTHS} | Seeds: {SEEDS} | Total: {TOTAL} scenarios")

In [ ]:
@kbench.task(name="zelusbench_attn_sustained_medium")
def zelusbench_attn_sustained_medium(llm) -> tuple[float, float]:
    _start = time.time()
    print(f"Running {TOTAL} scenarios...")
    print("=" * 60)

    all_scores = []
    depth_scores = {}
    scenario_num = 0

    for depth in DEPTHS:
        depth_scores[depth] = []
        for i in range(SEEDS):
            scenario_num += 1
            bg_rng = random.Random(i * 7919)
            cfg = ScenarioConfig.randomize_except(bg_rng, pinned={
                "min_chain_depth": depth,
                "max_chain_depth": depth,
                "dag_structure": DAGStructure.LINEAR,
                "query_target_depth": depth,
                "num_queries": 3,
                "seed": depth * 1000 + i,
            })
            gen = ScenarioGenerator(cfg)
            scenario = gen.generate(f"sustained_{depth}_{i}")

            response = llm.prompt(scenario.prompt)
            scores = score_response(response, scenario)
            all_scores.extend(scores)

            avg = float(np.mean([s.score for s in scores]))
            depth_scores[depth].append(avg)

            q_depths = [s.chain_depth for s in scores]
            tiers = [s.tier.name for s in scores]
            bg = f"dim={cfg.dim} dist={cfg.distractor_level.name} tx={cfg.transform_density.name}"
            print(f"  [{scenario_num}/{TOTAL}] depth={depth} i={i} "
                  f"avg={avg:.2f} q_depths={q_depths} tiers={tiers} | {bg}")

        depth_avg = float(np.mean(depth_scores[depth]))
        print(f"  >> Depth {depth} mean: {depth_avg:.3f}")

    print("\n" + "=" * 60)
    print("RESULTS BY DEPTH:")
    for depth in DEPTHS:
        avg = float(np.mean(depth_scores[depth]))
        print(f"  depth={depth:2d}  accuracy={avg:.3f}")
        kbench.assertions.assert_true(
            avg >= 0,
            expectation=f"Depth {depth}: valid answers (got {avg:.3f})"
        )

    overall = float(np.mean([s.score for s in all_scores]))
    std = float(np.std([s.score for s in all_scores]))
    elapsed = time.time() - _start
    print(f"\nOverall: {overall:.3f} +/- {std:.3f} | {len(all_scores)} queries | {elapsed:.1f}s")
    return overall, std


zelusbench_attn_sustained_medium.run(llm=kbench.llm)

In [ ]:
%choose zelusbench_attn_sustained_medium